# Xiangqi AI — Unified Bronze → Silver Data Pipeline

**Một notebook dùng chung cho cả CCPD (PGN, Big5) và Alpha-Beta (JSONL).**

- Không tải dữ liệu từ internet hoặc ingest lại Bronze: notebook đọc các file đã có tại `XiangqiAI/data/01_bronze`.
- CCPD → `scripts.build_ccpd_silver` → `CCPDPipeline` (parse PGN, replay hợp lệ, deduplicate, quarantine).
- Alpha-Beta → `scripts.build_silver` → `SilverPipeline.run_alphabeta` (parse JSONL, replay, deduplicate, quarantine).
- Kết quả vào `XiangqiAI/data/02_silver/<source>_<mode>_<run_id>`, mỗi source có report riêng, đồng bộ qua `.uploading` và kiểm SHA-256; không ghi đè run cũ.
- `MODE="sample"` là thử nghiệm. `MODE="full"` dùng toàn bộ dữ liệu đầu vào (CCPD 595 PGN nếu Bronze đầy đủ).
- Giữ nguyên bytes Bronze và metadata provenance; Silver chưa phải Gold/training.

Đối với Alpha-Beta, sample giới hạn dòng **trong CLI** qua `--max-records`, không sinh file JSONL mẫu nên SHA provenance khớp Bronze gốc.

Dùng `SOURCES = ["ccpd", "alphabeta"]` nếu muốn chạy **cả hai** trong một lần Run all.

## 1. Mount Google Drive và cấu hình

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/XiangqiAI")
BRONZE_ROOT = DRIVE_PROJECT / "data" / "01_bronze"
SILVER_ROOT = DRIVE_PROJECT / "data" / "02_silver"
REPO_DIR = Path("/content/ML-Advance-Group")
BACKEND_DIR = REPO_DIR / "project" / "backend"
LOCAL_ROOT = Path("/content/xiangqi_silver_runs")

# Chọn một hoặc nhiều nguồn; thứ tự trong list là thứ tự xử lý.
SOURCES = ["ccpd"]           # ["alphabeta"] hoặc ["ccpd", "alphabeta"]
MODE = "full"               # "sample" hoặc "full"
SAMPLE_GAMES = 20
RANDOM_SEED = 42
EXPECTED_CCPD_FILES = 595
REQUIRE_COMPLETE_CCPD = True
ALPHABETA_FILES = [
    "alphabeta_depth3_book_topk3_dedup_v2_seed42_max240.jsonl",
]
GIT_BRANCH = "main"

if not SOURCES or len(set(SOURCES)) != len(SOURCES):
    raise ValueError("SOURCES must contain unique source names")
if any(name not in {"ccpd", "alphabeta"} for name in SOURCES):
    raise ValueError("SOURCES supports only 'ccpd' and 'alphabeta'")
if MODE not in {"sample", "full"} or SAMPLE_GAMES < 1:
    raise ValueError("Invalid MODE/SAMPLE_GAMES")
print("Selected sources:", SOURCES, "| mode:", MODE)

## 2. Cập nhật backend

Cài đặt package của repository và dùng đúng CLI theo nguồn dữ liệu. Không chứa logic parse/replay của các source trong notebook.

In [ ]:
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", GIT_BRANCH,
        "https://github.com/Ko4rut/ML-Advance-Group.git", str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", GIT_BRANCH], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(BACKEND_DIR)],
    check=True,
)
REVISION = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
for script in ("build_ccpd_silver.py", "build_silver.py"):
    if not (BACKEND_DIR / "scripts" / script).is_file():
        raise FileNotFoundError(f"Missing backend CLI: {script}")
print("Backend Git SHA:", REVISION)

## 3. Kiểm tra Bronze và chọn dữ liệu

- **CCPD:** chỉ chọn file `.pgn` ở Bronze, loại LICENSE/README/manifests. Sample ngẫu nhiên có seed cố định; full yêu cầu đủ 595 PGN nếu `REQUIRE_COMPLETE_CCPD=True`.
- **Alpha-Beta:** chọn file JSONL theo `ALPHABETA_FILES`; sample vẫn tải **nguyên file** và dùng `--max-records`. Loại file trống và checkpoint.
- Các bản ghi nguồn luôn phải có `<filename>.manifest.json` đúng SHA-256. Nếu nguồn bị thay đổi, dừng trước khi tạo output.

In [ ]:
import hashlib
import json
import random
from datetime import datetime, timezone

from src.data.bronze.models import BronzeFileManifest

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for part in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(part)
    return digest.hexdigest()

def read_manifest(path, source):
    sidecar = path.with_name(path.name + ".manifest.json")
    if not sidecar.is_file():
        raise FileNotFoundError(f"Missing Bronze manifest: {sidecar}")
    manifest = BronzeFileManifest.model_validate_json(
        sidecar.read_text(encoding="utf-8")
    )
    if manifest.source != source or manifest.file_sha256 != sha256_file(path):
        raise ValueError(f"Bronze manifest/source/checksum mismatch: {path}")
    return manifest

source_files = {}
for source_name in SOURCES:
    folder = BRONZE_ROOT / source_name
    if not folder.is_dir():
        raise FileNotFoundError(f"Bronze folder missing: {folder}")

    if source_name == "ccpd":
        available = sorted(folder.glob("*.pgn"))
        if not available:
            raise ValueError("Bronze CCPD has no PGN files")
        if MODE == "full" and REQUIRE_COMPLETE_CCPD:
            if len(available) != EXPECTED_CCPD_FILES:
                raise ValueError(
                    f"CCPD full requires {EXPECTED_CCPD_FILES} PGNs, "
                    f"but Bronze has {len(available)}"
                )
        if MODE == "sample":
            selected = sorted(random.Random(RANDOM_SEED).sample(
                available, min(SAMPLE_GAMES, len(available))
            ))
        else:
            selected = available
    else:
        selected = [folder / filename for filename in ALPHABETA_FILES]
        for path in selected:
            if not path.is_file() or path.stat().st_size == 0:
                raise FileNotFoundError(f"Missing or empty Alpha-Beta file: {path}")
        if len({path.name for path in selected}) != len(selected):
            raise ValueError("Duplicate ALPHABETA_FILES")

    # Validate all selected files before staging.
    for file in selected:
        read_manifest(file, source_name)
    source_files[source_name] = selected
    print(source_name, "selected files:", len(selected))

## 4. Stage bản sao Bronze trên /content

Ghi toàn bộ file gốc + manifest vào workspace tạm Colab. Đối chiếu lại SHA-256 sau khi sao chép. Không thay đổi Bronze trong Drive.

Đối với Alpha-Beta sample, **không rút gọn JSONL**: chỉ giới hạn bản ghi ở CLI, tránh sai lệch provenance.

In [ ]:
import shutil

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
runs = {}
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

for source_name in SOURCES:
    run_id = f"{source_name}_{MODE}_{timestamp}"
    run_root = LOCAL_ROOT / run_id
    local_bronze = run_root / "bronze"
    local_silver = run_root / "silver"
    local_bronze.mkdir(parents=True, exist_ok=False)

    copied_files = []
    for source_file in source_files[source_name]:
        local_file = local_bronze / source_file.name
        sidecar = source_file.with_name(source_file.name + ".manifest.json")
        shutil.copy2(source_file, local_file)
        shutil.copy2(sidecar, local_file.with_name(local_file.name + ".manifest.json"))
        if sha256_file(local_file) != sha256_file(source_file):
            raise IOError(f"Failed staging checksum verification: {source_file}")
        read_manifest(local_file, source_name)
        copied_files.append(local_file)
    runs[source_name] = {
        "run_id": run_id,
        "input_files": copied_files,
        "output": local_silver,
    }
    print(source_name, "staged:", len(copied_files), "files")

## 5. Chạy Silver theo từng source

- CCPD gọi `scripts.build_ccpd_silver`, có thể xử lý Big5 PGN, kiểm SHA manifest và ghi report.
- Alpha-Beta gọi `scripts.build_silver`, `--max-records` chỉ dùng cho sample. File Bronze giữ nguyên hash.
- Sau mỗi luồng, kiểm tra quy tắc `source_records = accepted + duplicate + quarantine` (CCPD dùng `source_files`).
- Nếu CLI lỗi, dừng; run chưa được đồng bộ lên Drive.

In [ ]:
from collections import Counter

for source_name in SOURCES:
    data = runs[source_name]
    command = [sys.executable, "-m"]
    if source_name == "ccpd":
        command += [
            "scripts.build_ccpd_silver",
            *[str(p) for p in data["input_files"]],
            "--output-dir", str(data["output"]),
        ]
    else:
        manifest = read_manifest(data["input_files"][0], source_name)
        version = manifest.source_version
        lic = manifest.license
        for file in data["input_files"][1:]:
            other = read_manifest(file, source_name)
            if other.source_version != version or other.license != lic:
                raise ValueError("Alpha-Beta files have differing source version/license")
        command += [
            "scripts.build_silver",
            *[str(p) for p in data["input_files"]],
            "--output-dir", str(data["output"]),
            "--source-version", version,
        ]
        if lic is not None:
            command += ["--license", lic]
        if MODE == "sample":
            command += ["--max-records", str(SAMPLE_GAMES)]

    result = subprocess.run(
        command, cwd=BACKEND_DIR, text=True, capture_output=True
    )
    if result.returncode != 0:
        print("STDOUT:", result.stdout[-4000:])
        print("STDERR:", result.stderr[-4000:])
        raise RuntimeError(f"Silver build failed for {source_name}")

    if source_name == "ccpd":
        metrics = json.loads((data["output"] / "report.json").read_text(encoding="utf-8"))
        total = metrics["source_files"]
    else:
        metrics = json.loads(result.stdout)
        total = metrics["source_records"]
        reasons = Counter()
        with (data["output"] / "quarantine.jsonl").open(encoding="utf-8") as stream:
            for line in stream:
                if line.strip():
                    reasons[json.loads(line)["error_code"]] += 1
        metrics["quarantine_error_codes"] = dict(reasons)

    if total != metrics["accepted_games"] + metrics["duplicate_games"] + metrics["quarantined_records"]:
        raise AssertionError(f"Silver totals inconsistent for {source_name}")
    metrics["source"] = source_name
    metrics["mode"] = MODE
    metrics["run_id"] = data["run_id"]
    metrics["git_revision"] = REVISION
    metrics["created_at"] = datetime.now(timezone.utc).isoformat()
    metrics["source_files_sha256"] = [
        {"file": file.name, "sha256": sha256_file(file)}
        for file in data["input_files"]
    ]
    (data["output"] / "report.json").write_text(
        json.dumps(metrics, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    data["report"] = metrics
    print(json.dumps({
        key: metrics[key]
        for key in ("source", "mode", "run_id", "accepted_games", "transitions",
                    "duplicate_games", "quarantined_records")
    }, ensure_ascii=False, indent=2))

## 6. Đối chiếu từng transition và đồng bộ Drive

Trước khi công bố output, kiểm tra số dòng `games.jsonl`, `transitions.jsonl`, `duplicates.jsonl`, `quarantine.jsonl` với report và số move của từng game.

Sau đó upload kết quả vào thư mục `.uploading`, kiểm SHA-256 file trên Drive, rồi đổi tên thành thư mục run chính thức. Run cũ không bị ghi đè.

In [ ]:
import os

REQUIRED_OUTPUT = (
    "games.jsonl", "transitions.jsonl", "duplicates.jsonl",
    "quarantine.jsonl", "report.json",
)
SILVER_ROOT.mkdir(parents=True, exist_ok=True)

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                yield json.loads(line)

for source_name in SOURCES:
    data = runs[source_name]
    out = data["output"]
    metrics = data["report"]
    missing = [name for name in REQUIRED_OUTPUT if not (out / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Missing Silver output files: {missing}")

    games = list(read_jsonl(out / "games.jsonl"))
    count_moves = sum(len(game["moves"]) for game in games)
    steps_count = sum(1 for _ in read_jsonl(out / "transitions.jsonl"))
    duplicate_count = sum(1 for _ in read_jsonl(out / "duplicates.jsonl"))
    quarantine_count = sum(1 for _ in read_jsonl(out / "quarantine.jsonl"))
    if (len(games), steps_count, count_moves, duplicate_count, quarantine_count) != (
        metrics["accepted_games"], metrics["transitions"], metrics["transitions"],
        metrics["duplicate_games"], metrics["quarantined_records"]
    ):
        raise AssertionError(f"Silver JSONL/report mismatch: {source_name}")

    target = SILVER_ROOT / data["run_id"]
    uploading = SILVER_ROOT / ("." + data["run_id"] + ".uploading")
    if target.exists() or uploading.exists():
        raise FileExistsError(f"Drive run directory already exists: {target}")
    shutil.copytree(out, uploading)
    for source_file in out.iterdir():
        if source_file.is_file():
            uploaded = uploading / source_file.name
            if sha256_file(source_file) != sha256_file(uploaded):
                raise IOError(f"Drive upload hash mismatch: {uploaded}")
    os.rename(uploading, target)
    data["drive_output"] = target
    print(f"{source_name}: validated and synced → {target}")

## Kết quả

- `SOURCE=["ccpd"]`: xử lý CCPD Bronze → Silver.
- `SOURCES=["alphabeta"]`: xử lý Alpha-Beta Bronze → Silver.
- `SOURCES=["ccpd", "alphabeta"]`: chạy **cả hai luồng**, mỗi luồng có run/report riêng.
- `MODE="sample"`: lấy 20 PGN cố định (CCPD), hoặc 20 bản ghi JSONL đầu tiên (Alpha-Beta), dùng để kiểm tra.
- `MODE="full"`: xử lý toàn bộ 595 PGN (nếu đủ trong Bronze) hoặc toàn bộ bản ghi Alpha-Beta đã chọn.
- Cả hai giữ nguyên Bronze; Silver chỉ có dữ liệu được validate/replay và deduplicate. Gold/Training chưa chạy.

Muốn chạy full 595 CCPD: đặt `SOURCES=["ccpd"]`, `MODE="full"` rồi **Runtime → Run all**.